# Notebook 1 — Kokoro v1.0 (Hindi)

- Repo: https://github.com/hexgrad/kokoro
- Model: https://huggingface.co/hexgrad/Kokoro-82M
- Voices: https://huggingface.co/hexgrad/Kokoro-82M/blob/main/VOICES.md
- Hindi voices: `hf_alpha`, `hf_beta` (female), `hm_omega`, `hm_psi` (male)

> Kokoro saw only ~6 hours of Hindi at training. Set expectations low for naturalness.
> If `lang_code='h'` rejects Roman-script Hinglish, **the error becomes the data point** —
> catch and log it; do NOT silently fall back to English.


In [ ]:
# === Cell 1: install ===
!pip install -q "kokoro>=0.9.4" soundfile
!apt-get -qq -y install espeak-ng > /dev/null

In [ ]:
# Mount Drive (or skip if running locally) and clone the audit folder.
# Adjust this cell to point AUDIT_DIR at wherever audit/ lives in your runtime.
import os
from pathlib import Path

# Two common patterns:
#   1. Colab + Drive: AUDIT_DIR = "/content/drive/MyDrive/hienglish/audit"
#   2. Colab + git clone:
#         !git clone https://github.com/<you>/hienglish.git /content/hienglish
#         AUDIT_DIR = "/content/hienglish/audit"
#   3. Local: AUDIT_DIR = str(Path.cwd().parent / "audit")  (if launched from notebooks/)

AUDIT_DIR = os.environ.get("AUDIT_DIR", "/content/audit")
assert Path(AUDIT_DIR).is_dir(), f"AUDIT_DIR={AUDIT_DIR} missing — set it before running."
print(f"AUDIT_DIR = {AUDIT_DIR}")


In [ ]:
import csv
from pathlib import Path

EVAL_TSV = Path(AUDIT_DIR) / "eval_sentences.tsv"
with open(EVAL_TSV, encoding="utf-8") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 30, f"expected 30 sentences, got {len(rows)}"
print(f"Loaded {len(rows)} sentences from {EVAL_TSV}")
print(rows[0])


In [ ]:
# === Cell 4: build pipelines ===
from pathlib import Path
import time, json
import numpy as np
import soundfile as sf
from kokoro import KPipeline

MODEL_NAME = "kokoro"
OUT = Path(AUDIT_DIR) / "results" / MODEL_NAME
OUT.mkdir(parents=True, exist_ok=True)

# Two pipelines: Hindi for everything Devanagari/Hindi-leaning,
# American English ONLY for english_with_NE category.
pipe_hi = KPipeline(lang_code="h")
pipe_en = KPipeline(lang_code="a")

VOICE_HI = "hf_alpha"   # female Hindi
VOICE_EN = "af_heart"   # female American English


In [ ]:
# === Cell 5: run inference, save audio + log ===
log = []
for r in rows:
    rid, cat, text = r["id"], r["category"], r["text"]
    t0 = time.time()
    try:
        if cat == "english_with_NE":
            pipe, voice = pipe_en, VOICE_EN
        else:
            # Hindi pipeline for ALL non-English categories — including pure_roman
            # and mixed_script. We want to see if it copes or breaks.
            pipe, voice = pipe_hi, VOICE_HI

        gen = pipe(text, voice=voice, speed=1.0)
        chunks, phonemes = [], []
        for gs, ps, audio in gen:
            chunks.append(audio); phonemes.append(ps)
        full = np.concatenate(chunks) if chunks else np.zeros(1, dtype=np.float32)
        out_path = OUT / f"{rid}.wav"
        sf.write(out_path, full, 24000)
        log.append({
            "id": rid, "category": cat, "voice": voice,
            "phonemes": " ".join(phonemes),
            "duration_s": float(len(full) / 24000),
            "elapsed_s": time.time() - t0,
            "status": "ok",
        })
    except Exception as e:
        log.append({"id": rid, "category": cat, "status": "error", "error": str(e)})
        print(f"  [error] {rid}: {e}")


In [ ]:
import json
out_log = Path(OUT) / "log.json"
with open(out_log, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)

n_ok = sum(1 for x in log if x["status"] == "ok")
print(f"{MODEL_NAME}: {n_ok}/30 succeeded — log at {out_log}")


In [ ]:
# === Cell 7: sanity check ===
n_ok = sum(1 for x in log if x["status"] == "ok")
assert n_ok >= 28, f"Too many failures: {30 - n_ok}. Inspect log.json before declaring done."
